# SEASONAL DATA ANALYSIS

In [1]:
import pandas as pd
import os
import win32com.client as win32
import math
import pythoncom
from win32com.client import Dispatch


def compute_seasonal_months(
    df,
    date_col="order_date",
    qty_col="sold_qty",
    require_date_col=True,
    month_name_fmt="%b",
    product_col="Product"   
):
    df = df.copy()

    if not (set(["Year","Month","Month_name"]).issubset(df.columns)):
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            df["Year"] = df[date_col].dt.year
            df["Month"] = df[date_col].dt.month
            df["Month_name"] = df[date_col].dt.strftime(month_name_fmt)
        else:
            if require_date_col:
                raise ValueError("Input df must contain Year/Month/Month_name or a valid date_col.")

    if qty_col not in df.columns:
        raise ValueError(f"Quantity column '{qty_col}' not found.")

    if product_col not in df.columns:
        raise ValueError(f"Product column '{product_col}' not found.")

    monthly = (
        df.groupby(["Year", "Month", "Month_name", product_col], as_index=False)[qty_col]
          .sum()
          .rename(columns={qty_col: "Total sold qty"})
    )

    
    monthly["Average"] = (
        monthly.groupby(["Year", product_col])["Total sold qty"]
               .transform("mean")
    )

    monthly["Average crossed month"] = monthly.apply(
        lambda r: r["Month_name"] if r["Total sold qty"] > r["Average"] else "",
        axis=1
    )

    
    def seasonal_logic(row, data):
        current_year = int(row["Year"])
        current_month_name = row["Month_name"]
        current_value = row["Total sold qty"]
        avg_crossed_month = row["Average crossed month"]

        if not avg_crossed_month:
            return pd.Series({"seasonal_months": "", "Seasonal sales value": None})

        same_product = data[product_col] == row[product_col]

        prev_rows = data[
            same_product &
            (data["Month_name"] == current_month_name) &
            (data["Year"] == current_year - 1) &
            (data["Average crossed month"] == current_month_name)
        ]
        if not prev_rows.empty:
            return pd.Series({"seasonal_months": current_month_name,
                              "Seasonal sales value": current_value})

        future_rows = data[
            same_product &
            (data["Month_name"] == current_month_name) &
            (data["Year"] > current_year) &
            (data["Average crossed month"] == current_month_name)
        ]
        if not future_rows.empty:
            return pd.Series({"seasonal_months": current_month_name,
                              "Seasonal sales value": current_value})

        return pd.Series({"seasonal_months": "", "Seasonal sales value": None})

    monthly = monthly.sort_values(["Year", "Month"]).reset_index(drop=True)
    seasonal_results = monthly.apply(seasonal_logic, axis=1, data=monthly)
    monthly[["seasonal_months", "Seasonal sales value"]] = seasonal_results

    cols_out = [
        "Year", "Month", "Month_name", product_col,
        "Total sold qty", "Average", "Average crossed month",
        "seasonal_months", "Seasonal sales value"
    ]
    return monthly[cols_out]


In [2]:
df = pd.read_excel("Cavins_Complete_Districtwise_Sales.xlsx", sheet_name="Data")

In [3]:

df = pd.read_excel("Cavins_Complete_Districtwise_Sales.xlsx", sheet_name="Data")
products = df["Product"].dropna().unique()

for p in products:
    df_p = df[df["Product"] == p].copy()

    if df_p.empty:
        print(f"❌ No rows for {p}")
        continue
    result = compute_seasonal_months(
        df_p,
        date_col="order_date",
        qty_col="sold_qty"
    )
    file_name = f"Seasonal_{p}.xlsx"
    result.to_excel(file_name, index=False)

    print(f"✅ Saved: {file_name}")


✅ Saved: Seasonal_Karthika Seeyakkai Powder.xlsx
✅ Saved: Seasonal_Cavin’s Milkshake – Chocolate.xlsx
✅ Saved: Seasonal_Cavin’s Curd.xlsx
✅ Saved: Seasonal_Cavin’s Paneer.xlsx
✅ Saved: Seasonal_Cavin’s Ghee.xlsx
✅ Saved: Seasonal_Cavin’s Buttermilk.xlsx
✅ Saved: Seasonal_Cavin’s Lassi.xlsx
✅ Saved: Seasonal_Cavin’s Milk – Toned.xlsx
✅ Saved: Seasonal_Cavin’s Milk – Full Cream.xlsx
✅ Saved: Seasonal_Cavin’s Paneer Cubes.xlsx


In [4]:

df = pd.read_excel("Cavins_Complete_Districtwise_Sales.xlsx", sheet_name="Data")

all_products_output = []


for product in df["Product"].dropna().unique():

    df_product = df[df["Product"] == product]

    seasonal_table = compute_seasonal_months(
        df_product,
        date_col="order_date",
        qty_col="sold_qty"
    )

    if seasonal_table.empty:
        continue

    all_products_output.append(seasonal_table)
final_output = pd.concat(all_products_output, ignore_index=True)
final_output.to_excel("All_Products_Seasonal_Output.xlsx", index=False)

print("✅ All products seasonal data saved in ONE Excel file (no duplicate Product column)")


✅ All products seasonal data saved in ONE Excel file (no duplicate Product column)


In [5]:
import pandas as pd

file_name = "All_Products_Seasonal_Output.xlsx"
df = pd.read_excel(file_name)
df["Month_year"] = df["Month_name"].astype(str) + "_" + df["Year"].astype(str)
cols = list(df.columns)
pos = cols.index("Month_name") + 1
cols.remove("Month_year")
cols.insert(pos, "Month_year")
df = df[cols]
df.to_excel(file_name, index=False)

print("✅ Month_year column added after Month_name successfully!")


✅ Month_year column added after Month_name successfully!


In [6]:


# ---------- Config ----------
INPUT_FILE = "All_Products_Seasonal_Output.xlsx"
OUTPUT_FILE = "All_Products_Seasonal_Perfect_Excel.xlsx"
CHART_LEFT_COL = "L"      # anchor column for chart
CHART_TOP_ROW = 3         # anchor row for chart
CHART_WIDTH = 600         # pixels approx
CHART_HEIGHT = 360        # pixels approx

HEADER_COLOR = (79, 129, 189)   # #4F81BD (R,G,B)
TITLE_FILL_COLOR = (79,129,189)
SEASON_MARKER_COLOR = (155,187,89)  # green-ish for marker
TOTAL_LINE_COLOR = (79,129,189)     # blue
AVERAGE_LINE_COLOR = (192,0,0)      # red
# ----------------------------

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

# read combined seasonal data
df = pd.read_excel(INPUT_FILE)

# ensure Month_year exists
if "Month_year" not in df.columns:
    if {"Month_name", "Year"}.issubset(df.columns):
        df["Month_year"] = df["Month_name"].astype(str) + "_" + df["Year"].astype(str)
    else:
        raise ValueError("Input must contain Month_year or Month_name+Year")

# helper to convert (R,G,B) to Excel RGB integer
def rgb_int(rgb):
    r,g,b = rgb
    return r + (g<<8) + (b<<16)

# start COM and Excel
pythoncom.CoInitialize()
excel = win32.gencache.EnsureDispatch("Excel.Application")
excel.Visible = False   # set True for debug to show Excel while running
excel.DisplayAlerts = False

# create a new workbook to hold sheets+charts
wb = excel.Workbooks.Add()
# remove default sheets
while wb.Sheets.Count > 1:
    wb.Sheets(1).Delete()

products = df["Product"].dropna().unique().tolist()

for product in products:
    prod_df = df[df["Product"] == product].copy()
    if prod_df.empty:
        continue

    # sort by Year, Month if available
    if {"Year","Month"}.issubset(prod_df.columns):
        prod_df = prod_df.sort_values(["Year","Month"]).reset_index(drop=True)

    # ensure Seasonal sales value exists and equals Total sold qty only where seasonal_months flagged
    if "Seasonal sales value" not in prod_df.columns:
        prod_df["Seasonal sales value"] = None

    for i in prod_df.index:
        flag = str(prod_df.at[i, "seasonal_months"]).strip()
        if flag not in ("", "nan", "None"):
            prod_df.at[i, "Seasonal sales value"] = prod_df.at[i, "Total sold qty"]
        else:
            prod_df.at[i, "Seasonal sales value"] = None

    # move Product to first column (no duplication)
    cols = list(prod_df.columns)
    if "Product" in cols:
        cols.remove("Product")
        cols = ["Product"] + cols
        prod_df = prod_df[cols]

    # create a new sheet
    sheet = wb.Sheets.Add()
    sheet.Name = str(product)[:31]  # max 31 chars

    # write header starting at row 2 (reserve row1 for optional title)
    start_row = 2
    for j, col in enumerate(prod_df.columns, start=1):
        sheet.Cells(start_row, j).Value = col
    # write data rows starting row 3
    for i, row in enumerate(prod_df.itertuples(index=False), start=start_row+1):
        for j, val in enumerate(row, start=1):
            if(isinstance(val, float) and math.isnan(val)) or pd.isna(val):
                sheet.Cells(i, j).Value = ""
            else:
                 sheet.Cells(i, j).Value = val

    last_row = start_row + len(prod_df)
    last_col = len(prod_df.columns)

    # format header (row 2)
    header_rgb = rgb_int(HEADER_COLOR)
    for j in range(1, last_col+1):
        cell = sheet.Cells(start_row, j)
        cell.Interior.Color = header_rgb
        cell.Font.Color = rgb_int((255,255,255))  # white text
        cell.Font.Bold = True
        cell.HorizontalAlignment = win32.constants.xlCenter
        cell.VerticalAlignment = win32.constants.xlCenter
        # add thin border
        cell.Borders.LineStyle = 1

    # add thin borders for all table cells header+data
    for r in range(start_row, last_row+1):
        for c in range(1, last_col+1):
            cell = sheet.Cells(r,c)
            # set border (Thin)
            for b in (win32.constants.xlEdgeLeft, win32.constants.xlEdgeTop,
                      win32.constants.xlEdgeBottom, win32.constants.xlEdgeRight):
                cell.Borders(b).LineStyle = 1

    # auto-fit columns
    for j in range(1, last_col+1):
        sheet.Columns(j).AutoFit()

    # prepare chart: create ChartObject at column L, row CHART_TOP_ROW
    # compute left/top in points: use Range to get left/top
    anchor_cell = sheet.Range(f"{CHART_LEFT_COL}{CHART_TOP_ROW}")
    left = anchor_cell.Left
    top = anchor_cell.Top
    chart_obj = sheet.ChartObjects().Add(left, top, CHART_WIDTH, CHART_HEIGHT)
    chart = chart_obj.Chart
    chart.ChartType = win32.constants.xlLineMarkers  # line with markers type initially

    # series source: we'll add series manually to control markers
    # X categories range (Month_year) -> data rows only
    # find index of columns
    headers = [sheet.Cells(start_row, c).Value for c in range(1, last_col+1)]
    def find_col(*names):
        for n in names:
            if n in headers:
                return headers.index(n) + 1
        return None

    month_col = find_col("Month_year", "Month Year", "Month_Year", "Month-Year")
    qty_col = find_col("Total sold qty", "Total sold", "Total_sold_qty")
    avg_col = find_col("Average", "Avg")
    seasonal_col = find_col("Seasonal sales value", "Seasonal_sales_value", "Seasonal sales")

    if not (month_col and qty_col and avg_col):
        print(f"Skipping chart for {product} because required columns not found.")
        continue

    # set category X values (list)
    x_range = sheet.Range(sheet.Cells(start_row+1, month_col), sheet.Cells(last_row, month_col))

    # remove any existing series
    while chart.SeriesCollection().Count > 0:
        chart.SeriesCollection(1).Delete()

    # add Total sold qty series
    total_series = chart.SeriesCollection().NewSeries()
    total_series.Name = "Total sold qty"
    total_series.Values = sheet.Range(sheet.Cells(start_row+1, qty_col), sheet.Cells(last_row, qty_col))
    total_series.XValues = x_range
    # hide markers for total: set MarkerStyle = -4142 (xlMarkerStyleNone)
    try:
        total_series.MarkerStyle = win32.constants.xlMarkerStyleNone
    except Exception:
        total_series.MarkerSize = 1  # fallback

    # color line (safe): use RGB integer
    try:
        total_series.Format.Line.ForeColor.RGB = rgb_int(TOTAL_LINE_COLOR)
    except Exception:
        pass

    # add Average series
    avg_series = chart.SeriesCollection().NewSeries()
    avg_series.Name = "Average"
    avg_series.Values = sheet.Range(sheet.Cells(start_row+1, avg_col), sheet.Cells(last_row, avg_col))
    avg_series.XValues = x_range
    try:
        avg_series.MarkerStyle = win32.constants.xlMarkerStyleNone
    except Exception:
        avg_series.MarkerSize = 1
    # dashed line (set DashStyle)
    try:
        avg_series.Format.Line.DashStyle = win32.constants.msoLineDash
        avg_series.Format.Line.ForeColor.RGB = rgb_int(AVERAGE_LINE_COLOR)
    except Exception:
        pass

    # add Seasonal marker series (markers only)
    if seasonal_col:
        season_series = chart.SeriesCollection().NewSeries()
        season_series.Name = "Seasonal Months"
        # Use Seasonal sales value column (it contains total sold qty at seasonal rows, blanks otherwise)
        season_series.Values = sheet.Range(sheet.Cells(start_row+1, seasonal_col), sheet.Cells(last_row, seasonal_col))
        season_series.XValues = x_range
        # remove line - we set no line and markers only
        try:
            season_series.Format.Line.Visible = False
        except Exception:
            pass
        # marker shape diamond
        try:
            season_series.MarkerStyle = win32.constants.xlMarkerStyleDiamond
            season_series.MarkerSize = 8
        except Exception:
            pass
        # marker color (fill)
        try:
            season_series.Format.Fill.ForeColor.RGB = rgb_int(SEASON_MARKER_COLOR)
        except Exception:
            pass

    # Set chart title and attempt to fill title background and color text
    chart.HasTitle = True
    chart.ChartTitle.Text = f"{product} Seasonal Data"
    
    # --- REMOVE GRIDLINES ---
    chart.Axes(win32.constants.xlCategory).HasMajorGridlines = False
    chart.Axes(win32.constants.xlCategory).HasMinorGridlines = False
    chart.Axes(win32.constants.xlValue).HasMajorGridlines = False
    chart.Axes(win32.constants.xlValue).HasMinorGridlines = False

    try:
        # set title font color
        chart.ChartTitle.Format.TextFrame2.TextRange.Font.Fill.ForeColor.RGB = rgb_int((255,255,255))
        # set title shape fill
        chart.ChartTitle.Format.Fill.ForeColor.RGB = rgb_int(TITLE_FILL_COLOR)
    except Exception:
        # ignore if not supported by host Excel version
        pass

    # place legend below
    chart.Legend.Position = win32.constants.xlLegendPositionBottom

    # axis titles
    try:
        chart.Axes(win32.constants.xlCategory).HasTitle = True
        chart.Axes(win32.constants.xlCategory).AxisTitle.Text = "Month_year"
        chart.Axes(win32.constants.xlValue).HasTitle = True
        chart.Axes(win32.constants.xlValue).AxisTitle.Text = "Total sold qty"
        # rotate x ticklabels (angled)
        chart.Axes(win32.constants.xlCategory).TickLabels.Orientation = -45
    except Exception:
        pass

# Save and close
wb.SaveAs(os.path.abspath(OUTPUT_FILE))
wb.Close()
excel.Quit()
pythoncom.CoUninitialize()
print("Done. Saved:", OUTPUT_FILE)


Done. Saved: All_Products_Seasonal_Perfect_Excel.xlsx
